# go-evo — Phase 2: evolving 10-layer MLP Go players on 19×19

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrimozRavbar/go-evo/blob/main/notebooks/phase2_colab.ipynb)

* 19×19 board, komi 7.5, move cap 3·N² = 1083.
* Each generation every genome plays **5 games** (as in phase 1).
* **Only winners** (more wins than losses) have children. Children = Gaussian mutation of one parent, **no crossover**. Winners survive (elitism).
* **Control:** identical runs where the "winners" are picked at random. If the real runs improve more than the controls, selection is doing something.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # or "claude/trusting-keller-qlgif5" until merged
if IN_COLAB:
    if not os.path.exists("go-evo"):
        !git clone -q -b {BRANCH} https://github.com/PrimozRavbar/go-evo.git
    %cd go-evo
    !pip install -q -r requirements.txt
else:
    os.chdir(os.path.abspath(".."))  # repo root
    sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from goevo import NetConfig, GameConfig, Player, play_game, load_population
from goevo.evolve import EvoConfig, run_evolution, load_history
from goevo.analysis import bench_curve, survivor_advantage, head_to_head, plot_runs
from goevo.viz import animate_game
print("CPUs:", os.cpu_count())

## A. Results of the pushed runs (`runs/`)
Four runs of 300 generations: `sel_s0`, `sel_s1` (selection) and `ctl_s0`, `ctl_s1` (random-parent control).

In [ ]:
RUNS = {k: f"runs/{k}" for k in ["sel_s0", "sel_s1", "ctl_s0", "ctl_s1"] if os.path.exists(f"runs/{k}/history.jsonl")}
plot_runs(RUNS, "/tmp/runs.png")
from IPython.display import Image, display
display(Image("/tmp/runs.png"))
for k, d in RUNS.items():
    h = load_history(d); g, m = bench_curve(h)
    print(f"{k}: bench win-rate gen0={m[0]:.3f}  last-5-avg={m[-5:].mean():.3f}  ({len(h)-1} generations)")

## B. Watch: evolved champion vs a generation-0 genome

In [ ]:
RUN = "runs/sel_s0"
ckpts = sorted(f for f in os.listdir(RUN) if f.endswith(".pt"))
print(ckpts)
final, net_cfg, names, extra = load_population(f"{RUN}/{ckpts[-1]}")
gen0, _, names0, _ = load_population(f"{RUN}/{ckpts[0]}")
game_cfg = GameConfig(board_size=net_cfg.board_size, backend="native")
champ = Player(final[0], net_cfg, name=f"final {names[0]}")   # winners are stored first (elitism)
old = Player(gen0[0], net_cfg, name=f"gen0 {names0[0]}")
rec = play_game(champ, old, game_cfg)
print(rec.black, "vs", rec.white, "->", rec.result_string, len(rec.moves), "moves")
animate_game(rec, interval_ms=60)

## C. Head-to-head: final population vs generation-0 population (all pairs, both colours = 200 games)

In [ ]:
wr, n = head_to_head(final, gen0, net_cfg, game_cfg, workers=os.cpu_count())
print(f"final vs gen0: win rate {wr:.3f} over {n} games")

## D. Run your own (short) evolution here

In [ ]:
evo = EvoConfig(pop_size=10, games_per_player=5, generations=10, sigma=0.1,
                selection="winners", bench_every=5, checkpoint_every=5,
                out_dir="runs/colab_demo", workers=os.cpu_count())
hist = run_evolution(evo, NetConfig(board_size=19), GameConfig(board_size=19, backend="native"))